<a href="https://colab.research.google.com/github/blaw05303/genre-scout/blob/main/Genre_Scout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Genre Scout ##
My app helps a small indie game studio decide which genre to make its next game in by comparing genres’ ratings, player counts, number of competing games, and typical prices in Steam’s most-reviewed games.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/4600_Project/


In [ ]:
from google import genai
from google.colab import userdata

key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)
MODEL = "gemini-flash-lite-latest"

In [7]:
def top_genres(games, n):
    games = games.copy()
    games["genre"] = games["genres"].str.split(",")
    games = games.explode("genre")
    games["genre"] = games["genre"].str.strip()
    best = games.groupby("genre")["pct_pos_total"].mean().sort_values(ascending=False)
    return best.head(n).round(1).reset_index()

def genre_commonalities(games, top):
    lines = []
    for genre in top["genre"]:
        # First 5 listed games that belong to this genre
        first5 = games[games["genres"].str.contains(genre, na=False)].head(5)
        for _, r in first5.iterrows():
            lines.append(f"{genre} | {r['name']} | {r['short_description'][:100]}")

    prompt = (
        "These are games from the top-rated Steam genres:\n"
        + "\n".join(lines)
        + "\n\nWhat do these games have in common? Do not include things like how popular they are or what genres they are from. Make each bullet point observation meaningfully different. Answer in 3 short bullet points."
    )
    try:
      response = client.models.generate_content(model=MODEL, contents=prompt)
      return response.text
    except Exception as e:
      return f"LLM request failed or timed out: {e}"

top = top_genres(games, 5)
print(top)
print(genre_commonalities(games, top))

        genre  pct_pos_total
0       Indie           89.6
1      Casual           88.6
2  Simulation           86.7
3   Adventure           86.2
4    Strategy           85.1
* **Multiplayer and Social Interaction:** Nearly every listed game emphasizes online cooperation or competition, bringing groups of players together either to survive, build, battle, or socialize.
* **Open-World and Sandbox Freedom:** Most titles drop players into expansive environments with minimal rigid structure, empowering them to set their own goals—whether that means designing contraptions, farming, or exploring ruins.
* **Survival and Resource Management:** A vast majority of these games require players to constantly gather materials, craft items, or manage vital resources in order to overcome hostile environments and stay alive.
